In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [3]:
POSBal = pd.read_csv('data/Internal/POS_CASH_balance.csv')

In [4]:
POSBal.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [6]:
POSBal.sort_values(by=['SK_ID_CURR','SK_ID_PREV','MONTHS_BALANCE'], inplace=True, ignore_index=True)

In [7]:
POSBal = POSBal.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [8]:
POSBal[POSBal['SK_ID_CURR']=='363269']

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
7382923,2404432,363269,-96,6.0,0.0,Active,379,0
7382924,2404432,363269,-95,6.0,0.0,Active,410,0
7382925,2404432,363269,-94,6.0,0.0,Active,440,0
7382926,2404432,363269,-93,6.0,0.0,Active,471,0
7382927,2404432,363269,-92,6.0,0.0,Active,501,0
...,...,...,...,...,...,...,...,...
7383053,2832518,363269,-5,30.0,15.0,Active,0,0
7383054,2832518,363269,-4,30.0,14.0,Active,0,0
7383055,2832518,363269,-3,30.0,13.0,Active,0,0
7383056,2832518,363269,-2,30.0,12.0,Active,0,0


In [ ]:
#check for installment prepayment

multiple installment counts

In [11]:
def get_active_count_last_year(x, df):
    """
    Count the number of active installments in the last year.
    """
    # Get the slice of the DataFrame relevant to the group x
    group_df = df.loc[x.index]
    
    # Create a boolean mask for the 'last year' condition
    last_year_mask = group_df['MONTHS_BALANCE'] > -12
    
    # Create a boolean mask for the 'Active' status condition
    active_mask = group_df['NAME_CONTRACT_STATUS'] == 'Active'
    
    return (last_year_mask & active_mask).sum()

In [ ]:
POSBal_Silver = POSBal.groupby(['SK_ID_CURR','SK_ID_PREV']).agg(
  #  POS_AVG_DURATION = ('MONTHS_BALANCE', 'min'),
  #  POS_FIRST_CREDIT = ('MONTHS_BALANCE', 'min'),
    POS_ACTUAL_DURATN = ('MONTHS_BALANCE', 'size'),
    POS_EXPECTED_DURATN =  ('CNT_INSTALMENT', 'max'),
    POS_INSTALMENT_CNT_FUTURE = ('CNT_INSTALMENT_FUTURE', 'min'),
    POS_ACTIVE_CNT_LAST_YEAR = ('MONTHS_BALANCE', lambda x: get_active_count_last_year(x, POSBal)),
    POS_MAX_DPD = ('SK_DPD_DEF', 'max'),
    POS_DAYS_IN_DPD = ('SK_DPD_DEF', lambda x: sum(x > 0)),
    POS_LAST_DPD_STATUS = ('SK_DPD_DEF', 'last'), # last DPD status of Previous ID
    POST_LAST_DPD_DAYS = ('MONTHS_BALANCE', 'last') # how many days ago was POS_LAST_DPD_STATUS
).reset_index()

Prepayment Flag is actual duration < originalexpected duration

In [ ]:
POSBal_Silver['PREPAY_COUNT'] = np.where(POSBal_Silver['POS_ACTUAL_DURATN'] < POSBal_Silver['POS_EXPECTED_DURATN'], 1, 0)

In [21]:
POSBal_Silver['DAYS_IN_DPD_TO_TOT_DURATION_RATIO'] = POSBal_Silver['POS_DAYS_IN_DPD'] / POSBal_Silver['POS_ACTUAL_DURATN']

In [18]:
POSBal_Silver.sort_values(by=['SK_ID_CURR','POST_LAST_DPD_DAYS'], inplace=True, ignore_index=True)

In [ ]:
POSBal_Gold = POSBal_Silver.groupby(['SK_ID_CURR']).agg(
    POS_PREV_ID_COUNT = ('SK_ID_PREV', lambda x: x.nunique()),

    POS_AVG_ACTUAL_DURATN = ('POS_ACTUAL_DURATN', 'mean'),
    POS_MAX_ACTUAL_DURATN = ('POS_ACTUAL_DURATN', 'max'),
    POS_MEDIAN_ACTUAL_DURATN = ('POS_ACTUAL_DURATN', 'median'),

    POS_AVG_INSTALMENT_CNT_FUTURE = ('POS_INSTALMENT_CNT_FUTURE', 'mean'),
    POS_MAX_INSTALMENT_CNT_FUTURE = ('POS_INSTALMENT_CNT_FUTURE', 'max'),
    POS_MEDIAN_INSTALMENT_CNT_FUTURE = ('POS_INSTALMENT_CNT_FUTURE', 'median'),
    
    POS_AVG_ACTIVE_CNT_LAST_YEAR = ('PREPAY_COUNT', 'mean'),
    POS_MAX_ACTIVE_CNT_LAST_YEAR = ('PREPAY_COUNT', 'max'),
    POS_MEDIAN_ACTIVE_CNT_LAST_YEAR = ('PREPAY_COUNT', 'median'),

    POS_MAX_MAX_DPD = ('POS_MAX_DPD', 'max'),    

    POS_AVG_DAYS_IN_DPD = ('POS_DAYS_IN_DPD', 'mean'),
    POS_MAX_DAYS_IN_DPD = ('POS_DAYS_IN_DPD', 'max'),
    POS_MEDIAN_DAYS_IN_DPD = ('POS_DAYS_IN_DPD', 'median'),
    POS_AVG_DAYS_IN_DPD_TO_TOT_DURATION_RATIO = ('DAYS_IN_DPD_TO_TOT_DURATION_RATIO', 'mean'),

    POS_LAST_DPD_STATUS = ('POS_LAST_DPD_STATUS', 'last')
    
).reset_index()